# Lab: Convolutional Neural Networks for Satellite Image Classification

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Learning Objectives
* A custom **Convolutional Neural Network** architecture is designed and implemented using PyTorch.
* **High-resolution satellite imagery** is mapped from fine-grained categories to broad macro-classes.
* **Classification model generalization** is evaluated through systematic **hyperparameter tuning**.
* Classification errors are analyzed by **tracing misclassified macro-classes back to their original micro-class spectral signatures**.
* Results and experimental trials are serialized utilizing Pydantic models for **standardized** evaluation.


In [20]:
# !pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aqibrehmanpirzada/nwpuresisc45")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1


In [21]:
import os
import json
from enum import Enum
import random
from typing import List, Dict, Any
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from pydantic import BaseModel
import warnings

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = device.cuda() if device.type == "cuda" else device
print("Using device:", device)

Using device: cpu


### Data Engineering and Macro-Class Mapping
The required NWPU-RESISC45 dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/aqibrehmanpirzada/nwpuresisc45). It consists of forty-five fine-grained classes. These are algorithmically mapped to four broad macro-classes. Cloud cover is categorized as unknown and must be filtered out during dataset initialization.

In [22]:
class LandCover(Enum):
    UNKNOWN = 0
    GREENERY = 1
    SAND = 2
    WATER = 3
    CEMENT_INFRASTRUCTURE = 4

macro_class_mapping = {
    'chaparral': LandCover.GREENERY, 
    'circular_farmland': LandCover.GREENERY, 
    'forest': LandCover.GREENERY,
    'golf_course': LandCover.GREENERY, 
    'meadow': LandCover.GREENERY, 
    'rectangular_farmland': LandCover.GREENERY, 
    'terrace': LandCover.GREENERY,

    'beach': LandCover.SAND, 
    'desert': LandCover.SAND, 
    'mountain': LandCover.SAND, 
    'island': LandCover.SAND,

    'harbor': LandCover.WATER, 
    'lake': LandCover.WATER, 
    'river': LandCover.WATER, 
    'sea_ice': LandCover.WATER, 
    'wetland': LandCover.WATER, 
    'ship': LandCover.WATER, 
    'snowberg': LandCover.WATER,

    'airport': LandCover.CEMENT_INFRASTRUCTURE, 
    'bridge': LandCover.CEMENT_INFRASTRUCTURE, 
    'church': LandCover.CEMENT_INFRASTRUCTURE, 
    'commercial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'dense_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'freeway': LandCover.CEMENT_INFRASTRUCTURE, 
    'industrial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'intersection': LandCover.CEMENT_INFRASTRUCTURE, 
    'medium_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'mobile_home_park': LandCover.CEMENT_INFRASTRUCTURE,
    'overpass': LandCover.CEMENT_INFRASTRUCTURE, 
    'palace': LandCover.CEMENT_INFRASTRUCTURE, 
    'parking_lot': LandCover.CEMENT_INFRASTRUCTURE, 
    'railway': LandCover.CEMENT_INFRASTRUCTURE,
    'railway_station': LandCover.CEMENT_INFRASTRUCTURE, 
    'roundabout': LandCover.CEMENT_INFRASTRUCTURE, 
    'runway': LandCover.CEMENT_INFRASTRUCTURE,
    'sparse_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'storage_tank': LandCover.CEMENT_INFRASTRUCTURE, 
    'thermal_power_station': LandCover.CEMENT_INFRASTRUCTURE,
    'baseball_diamond': LandCover.CEMENT_INFRASTRUCTURE, 
    'basketball_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'ground_track_field': LandCover.CEMENT_INFRASTRUCTURE,
    'stadium': LandCover.CEMENT_INFRASTRUCTURE, 
    'tennis_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'airplane': LandCover.CEMENT_INFRASTRUCTURE,

    'cloud': LandCover.UNKNOWN
}

# 0-based label mapping for CrossEntropyLoss
# Greenery: 0, Sand: 1, Water: 2, Cement/Infrastructure: 3
label_mapping = {
    LandCover.GREENERY: 0,
    LandCover.SAND: 1,
    LandCover.WATER: 2,
    LandCover.CEMENT_INFRASTRUCTURE: 3,
    LandCover.UNKNOWN: 4
}
CLASS_NAMES = ['GREENERY', 'SAND', 'WATER', 'CEMENT/INFRASTRUCTURE', 'UNKNOWN']
NUMBER_OF_CLASSES = len(CLASS_NAMES)

print("Macro class mapping defined with", NUMBER_OF_CLASSES, "classes:", CLASS_NAMES)

Macro class mapping defined with 5 classes: ['GREENERY', 'SAND', 'WATER', 'CEMENT/INFRASTRUCTURE', 'UNKNOWN']


### Dataset Initialization and Loading
The provided dataset directory is natively partitioned into training and testing subsets. Images must be loaded and mapped to their respective macro-classes, ensuring instances mapped to the `UNKNOWN` category are explicitly excluded. The testing subset is to be preserved strictly for final evaluation. To facilitate hyperparameter tuning without data leakage, the training subset must be further partitioned to generate an independent validation subset, or a cross-validation strategy must be implemented.

In [23]:

# Define dataset path and directories
path = "C:\\Users\\Rufai\\.cache\\kagglehub\\datasets\\aqibrehmanpirzada\\nwpuresisc45\\versions\\1"
dataset_path = os.path.join(path, "Dataset")
print("Dataset path:", dataset_path)
path_to_train_dir = os.path.join(dataset_path, "train")
print("Path to training directory:", path_to_train_dir)
path_to_test_dir = os.path.join(dataset_path, "test")
print("Path to testing directory:", path_to_test_dir)

IMG_SIZE = 64
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# filter UNKNOWN class from dataset
def filter_unknown_class(dataset):
    indices = [i for i, (_, label) in enumerate(dataset) if label != label_mapping[LandCover.UNKNOWN]]
    return Subset(dataset, indices)

Dataset path: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset
Path to training directory: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset\train
Path to testing directory: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset\test


In [24]:
# A transform pipeline is required (e.g., resizing, tensor conversion, normalization).
# train_dataset = ImageFolder(root='path_to_train_dir', transform=...)
# test_dataset = ImageFolder(root='path_to_test_dir', transform=...)

# Valid indices and mapped targets are to be extracted for both datasets, excluding the UNKNOWN class.
# The train_dataset must be dynamically split into distinct training and validation subsets.
# DataLoaders for the training, validation, and testing subsets are to be instantiated here.


### Convolutional Neural Network Architecture
A custom CNN class is defined. Spatial dimensions must be carefully tracked between convolutional outputs and fully connected layer inputs.

As a reference model, the LeNet-5 architecture illustrates the foundational principles of convolutional networks. The input tensor is sequentially processed through alternating convolutional and pooling layers to extract spatial features and reduce dimensionality. The resulting feature maps are subsequently flattened and passed through fully connected layers to yield the final classification probabilities.


In [25]:
class SatelliteCNN(nn.Module):
    def __init__(self):
        super(SatelliteCNN, self).__init__()
        # Convolutional layers, pooling layers, and fully connected layers are defined here.
        pass

    def forward(self, x):
        # The forward pass is implemented here.
        return x


### Model Training
The loss function and optimizer are initialized. The training loop is executed, updating model weights while validating performance on the held-out validation set to monitor for overfitting.

In [26]:
# Model, Loss Criterion, and Optimizer are instantiated here.
# The training loop over defined epochs is executed here.


### Quantitative Evaluation
The trained model is evaluated against the test subset. Overall Accuracy, Macro-Averaged F1 Scores, and the Confusion Matrix are computed.

In [27]:
# Predictions on the test set are generated.
# accuracy_score, f1_score, and confusion_matrix are computed and printed.


### Hyperparameter Tuning Analysis
Observations regarding hyperparameter tuning are required. Multiple trials must be recorded, documenting the chosen parameters, resultant accuracy, class-specific F-scores, and drawn conclusions. These records are to be serialized using the provided Pydantic schema in the submission cell.


In [28]:
# Hyperparameter tuning execution is to be implemented here.
# Utilizing a grid search strategy (e.g., via itertools.product or sklearn.model_selection.ParameterGrid) is suggested to systematically explore combinations of learning rates, batch sizes, optimizers, and epochs.


### Error Analysis and Confusion Patterns
The macro-class exhibiting the lowest accuracy must be identified based on the computed F-scores and Confusion Matrix. Subsequently, the original, fine-grained micro-classes constituting the misclassified instances within this macro-class are to be examined. Patterns explaining why the convolutional neural network struggles to differentiate these specific land cover categories must be formulated and reported in the submission payload.


In [29]:
# Error analysis execution is to be implemented here.
# A confusion matrix or cross-tabulation mapping the true original micro-classes against the predicted macro-classes for the least accurate macro-class must be generated here to facilitate pattern recognition.


### Pydantic Models and Submission Export
The `pydantic` library is utilized to ensure structural integrity of the final submission. The models below define the exact schema expected for automated grading. The variables are to be populated with the experimental findings, and the cell executed to write the JSON output.

In [30]:
class TrialRecord(BaseModel):
    hyperparams: Dict[str, Any]
    accuracy: float
    F_score_class: Dict[str, float]
    observations: List[str]
    conclusions: List[str]

class LabSubmission(BaseModel):
    student_ids: List[str]
    trials: List[TrialRecord]
    micro_class_error_analysis: str

# The records are populated here based on experimental iterations.
trial_one = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "Adam", "epochs": 10},
    accuracy=0.0,
    F_score_class={"GREENERY": 0.0, "SAND": 0.0, "WATER": 0.0, "CEMENT_INFRASTRUCTURE": 0.0},
    observations=["Initial training phase completed.", "Validation loss fluctuated."],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

# Additional trials are to be added to the list below.
submission = LabSubmission(
    student_ids=[9000000, 9000001],
    trials=[trial_one],
    micro_class_error_analysis="Insert the detailed analysis required by Question D here, discussing the specific original micro-classes that were misclassified within the worst-performing macro-class."
)

# The Pydantic model is serialized to JSON and saved for grading.
output_json = submission.model_dump_json(indent=4)
print(output_json)

with open("student_submission.json", "w") as f:
    f.write(output_json)


ValidationError: 2 validation errors for LabSubmission
student_ids.0
  Input should be a valid string [type=string_type, input_value=9000000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
student_ids.1
  Input should be a valid string [type=string_type, input_value=9000001, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

### Submission Protocol

The final deliverable **must be a zipped archive** submitted via Google Classroom. 

* The archive must contain the executed Jupyter Notebook and the generated `student_submission.json` file.
* The zipped file must be explicitly named utilizing the display name of the submitter exactly as it appears on Google Classroom.
* **Failure to adhere to these archival and naming conventions disrupts automated grading pipelines and will result in direct grade penalties.**

---

### Grading Rubric (Total: Ten Marks)

The requirement submission is evaluated based on the following criteria:

* **Data Engineering and Macro-Class Mapping (Two Marks)**
  The dataset is correctly loaded, successfully mapped to the defined Enum macro-classes, and securely partitioned into distinct training, validation, and testing subsets without data leakage.

* **Architecture and Training Execution (Two Marks)**
  A functional Convolutional Neural Network is constructed with appropriate spatial dimension tracking, and the training loop successfully computes gradients and updates model weights.

* **Hyperparameter Tuning (Two Marks)**
  Systematic hyperparameter tuning is executed, and multiple experimental trials are accurately serialized within the final JSON payload.

* **Error Analysis and Custom Confusion Matrix (Two Marks)**
  A custom confusion matrix mapping true micro-classes against predicted macro-classes is successfully generated, and systematic misclassification patterns for the least accurate class are logically analyzed in the written submission.

* **Quantitative Evaluation and Peer Comparison (Two Marks)**
  The macro-averaged F-score from the final experimental trial in the submitted list is explicitly reported and analytically compared against peer performances to identify relative architectural strengths and areas for optimization.
